<table style="width:100%; background-color: white; padding: 10px; border-radius: 6px; box-shadow: 0 0 5px rgba(0,0,0,0.2);">
  <tr>
    <td>
      <h1 style="margin-bottom: 0; color: black; font-size: clamp(1.5rem, 2.5vw, 2.5rem);">
        Drone RL with LLM-Guided Curriculum Learning
      </h1>
      <p style="margin-top: 8px; color: black; font-size: 1rem;">
        EVA Deep Reinforcement Learning &ndash; DRL Project, Spring 2026<br>
        Rino M. Albertin  
      </p>
    </td>
    <td align="right">
      <img src="docs/images/OST_Logo_DE_RGB@2000ppi.png" alt="OST Logo" width="250">
    </td>
  </tr>
</table>

<div style="display: flex; gap: 24px; align-items: flex-start; width: 100%;">

<div style="flex: 1.15; min-width: 0;">

## Introduction

This project investigates reinforcement learning for single-drone trajectory tracking in simulation. The goal is to train a drone policy that follows reference trajectories such as hover targets, straight lines, vertical movements and more complex paths.

The environment is based on PyBullet and `gym-pybullet-drones`. A Proximal Policy Optimization (PPO) agent is trained and evaluated on trajectory-tracking tasks. The main focus of the project is not only whether PPO can learn one fixed trajectory, but how different training strategies affect learning and generalization.

The project compares three training approaches:

1. direct PPO training,
2. a manually designed curriculum,
3. an LLM-guided curriculum.

The LLM is used only for curriculum generation, not for direct drone control.

</div>

<div style="flex: 0.85; min-width: 0;">
  <img src="./docs/media/hard_scenario_llm_curriculum_directrpm_dynprev_m-taskdist_medium_seed0_scenario_rollout.gif"
       alt="Example drone trajectory tracking rollout"
       style="width: 100%; height: auto; display: block;">
</div>

</div>

## Objective and Approach

The objective is to compare how different training strategies influence the tracking performance of a single-drone reinforcement learning agent.

The central research question is:

> Can curriculum learning improve PPO-based drone trajectory tracking compared to direct training, and can a local LLM be used as a safe curriculum task proposer?

The project compares three training approaches. The first baseline is direct PPO training, where the agent is trained directly on the target task distribution. The second approach is a manually designed curriculum, where the agent first learns simpler stabilization and movement tasks before progressing to harder tracking tasks. The third approach is an LLM-guided curriculum. In this setup, the first stage is a deterministic hover bootstrap, while later stages are proposed by a local LLM.

The LLM is not used as a controller. It does not send actions to the drone and it does not generate executable simulation code. Its role is limited to proposing structured curriculum tasks. These proposals are only accepted if they pass deterministic validation.

The main comparison is based on the tracking performance after training. The most important evaluation aspects are tracking error, termination behaviour, failure classification and qualitative trajectory plots or GIFs.

## Environment and Agent

The agent is trained in a single-drone trajectory-tracking environment. The environment wraps a PyBullet drone simulation and exposes a compact Gymnasium-compatible reinforcement learning interface.

At each step, the agent observes the drone state and the current reference position. The policy then outputs an action that is applied to the simulated drone. The reward penalizes tracking error and action effort.

### State space S

The observation contains:

| Component | Meaning |
|---|---|
| Current position | Current drone position in x, y and z direction |
| Reference position | Desired reference position at the current trajectory step |
| Position error | Difference between current and reference position |
| Trajectory progress | Normalized progress through the trajectory |
| Dynamics observation | Velocity, attitude and angular velocity |
| Previous action | Previous PPO-facing action |

The dynamics observation and previous-action observation are enabled in the main training configurations. This gives the policy more information about the physical state of the drone and its recent control behaviour.

### Action space A

Two action interfaces are implemented:

| Interface | Description |
|---|---|
| `pid_position` | PPO outputs normalized target-position commands. These are mapped to PID target positions. |
| `direct_rpm` | PPO outputs normalized motor commands that are mapped to motor RPM values. |

The `pid_position` interface is the main learning interface because it provides a higher-level and more stable control problem. The `direct_rpm` interface is more direct and therefore harder, because the policy has to learn lower-level motor behaviour.

## Reward Design

The reward is designed for trajectory tracking. It penalizes the Euclidean distance between the current drone position and the reference position. In addition, it penalizes the squared magnitude of the applied action to discourage unnecessarily large control commands.

The implemented reward has the form:

$$
r_t = - w_p \cdot \lVert p_t - p_t^{ref} \rVert_2 - w_a \cdot \lVert a_t \rVert_2^2
$$

where:  
- $p_t$: Current drone position  
- $p_t^{ref}$: Reference position at the current trajectory step  
- $a_t$: Applied action passed to the environment  
- $w_p$: Position error weight  
- $w_a$: Action cost weight  

In the current implementation, the default weights are:

$$
w_p = 1.0, \qquad w_a = 0.01
$$

The reward is therefore highest when the drone stays close to the reference trajectory and uses small control actions. Large tracking errors and large actions both reduce the reward.

## Algorithm

The project uses Proximal Policy Optimization (PPO) from Stable-Baselines3.

PPO is a policy optimization method. Instead of learning an explicit action-value table or Q-function and selecting the action with the highest Q-value as in Q-learning approaches, PPO directly optimizes a parameterized policy. The policy maps the current observation to an action distribution from which actions are sampled during training. 

The implementation follows an actor-critic structure. The actor represents the policy and predicts the action distribution for the current observation. The critic estimates the value of the current state. This value estimate is used to compute advantages, which measure whether an action performed better or worse than expected. PPO updates the policy using a clipped objective. The clipping limits how much the new policy is allowed to deviate from the previous policy in one update step. This makes training more stable than an unconstrained policy-gradient update, which is important for continuous-control tasks such as drone trajectory tracking.

The model used for prediction is a multilayer perceptron policy (`MlpPolicy`). The default architecture uses two hidden layers with 128 neurons each for the actor and critic networks. During training, the policy predicts actions stochastically to allow exploration. During evaluation, actions are predicted deterministically to measure the learned tracking behaviour consistently.

PPO was selected because it is a stable and widely used baseline algorithm for continuous-control reinforcement learning tasks. It is well suited for this project because the drone action space is continuous and the goal is to learn a smooth control policy for trajectory tracking.

## Training Setup

Training is performed from configuration files, not manually inside the notebook. This keeps the experiments reproducible and separates long-running training from the final report.

The main PPO setup uses:

| Parameter | Value |
|---|---:|
| Algorithm | PPO |
| Policy | MlpPolicy |
| Learning rate α | 0.0003 |
| Discount factor γ | 0.99 |
| GAE λ | 0.95 |
| PPO clip range | 0.2 |
| Entropy coefficient | 0.001 |
| Number of vectorized environments | 4 |
| Training timesteps per main run | 500,000 |
| Evaluation steps | 240 |

The main experiments use vectorized environments to collect PPO rollouts more efficiently. Each environment is seeded deterministically.

## Experiments

The experiments are grouped into three main comparisons. First, selected PPO settings are varied to study their effect on learning. Second, the two action interfaces are compared. Third, the main training strategies are evaluated.

An important part of the experimental design is how training tasks are sampled. The direct PPO baselines are not trained on one fixed trajectory only. Instead, they use randomized task distributions. In the `tracking_medium` setup, a new task is sampled at the beginning of each episode and then kept fixed during that episode. This exposes the policy to a range of trajectory types instead of letting it memorize one reference path.

The `tracking_medium` distribution contains several task families:

- stabilization tasks, such as hover and takeoff stabilization,
- basic tracking tasks, such as straight lines and start-hold-then-line movements,
- piecewise-linear paths, such as polylines, L-shapes, rectangles and squares,
- curved paths, such as circles, ellipses and figure-eight trajectories,
- altitude-changing tasks, such as vertical up-down, angled vertical, delayed-altitude polylines and multi-height polylines.

This creates a training distribution that still focuses on basic trajectory tracking, but also exposes the policy to more varied trajectory shapes. In addition to the `tracking_medium` runs, two `basic_show` direct PPO baselines are included. These runs are trained on one single connected show-style trajectory instead of a randomized task distribution. They are therefore not interpreted as general tracking baselines, but as task-specific reference runs. They are useful for illustrating how a policy can specialize to one coherent movement sequence, but this specialization does not necessarily transfer to the fixed generalization suite or to the OOD scenarios.

The curriculum approaches use a different structure. They do not immediately train on the full `tracking_medium` distribution. Instead, they progress through easier or more focused stages. Each stage still contains variation, for example in target position, movement length, direction, duration or height. This means that the curriculum stages are not simple repetitions of one identical trajectory, but controlled task distributions with a narrower difficulty range. In the manual curriculum, this progression is fixed in advance. The stages move from hover stabilization to vertical movements, short line movements, polylines and finally the broader medium tracking distribution. In the LLM curriculum, the first stage is a deterministic hover-bootstrap stage. Later stages are proposed by the local LLM, but only from structured and validated task definitions or known task distributions.

### 1. PPO Variant Comparison

A limited PPO variant comparison is included. The goal is not to perform a full hyperparameter sweep, but to compare selected settings that are relevant for training stability and tracking performance.

| Variant | Changed setting | Purpose |
|---|---|---|
| Default PPO | Baseline PPO configuration | Reference setting for all comparisons |
| `net256` | Larger policy and value network | Test whether a larger network improves tracking |
| `low_lr` | Lower learning rate $\alpha = 0.0001$ | Test whether slower policy updates improve stability |
| `gamma095` | Lower discount factor $\gamma = 0.95$ | Test whether a shorter effective planning horizon changes tracking performance |
| `ent005` | Higher entropy coefficient | Encourage more exploration |
| `clip010` | Smaller PPO clip range | Make policy updates more conservative |
| `targetkl015` | Lower target KL limit | Stop overly large policy updates earlier |

### 2. Action-Interface Comparison

The project compares `pid_position` and `direct_rpm`.

The `pid_position` interface is expected to be easier because the policy outputs target-position commands, while the lower-level PID controller handles part of the stabilization. This makes it the main interface for the report.

The `direct_rpm` interface is expected to be harder because the policy outputs motor-level commands. This gives the agent more direct control, but also makes the learning problem less stable.

### 3. Training Strategy Comparison

The main method comparison evaluates direct PPO, manual curriculum learning and LLM-guided curriculum learning.

Direct PPO is the baseline trained directly on the target task distribution. It therefore sees a broader task variety from the beginning of training.

The manual curriculum follows a fixed progression from easier to harder task distributions. This gives the agent a more structured learning path, but the progression itself is hand-designed.

The LLM curriculum also starts with a simple hover-bootstrap stage. After that, a local LLM proposes the next curriculum stages based on previous training feedback. These proposals are not used directly. They are accepted only if they pass deterministic validation before training.

## Evaluation Protocol

The trained policies are evaluated separately from training. Evaluation is based on deterministic rollout settings and fixed task definitions.

The evaluation answers three questions. Own-task evaluation tests how well the policy performs on its representative training task. Generalization evaluation tests how well the policy performs on fixed tasks outside the exact training episode. Scenario evaluation tests how the policy behaves in selected visual scenarios.

The most important quantitative metrics are:

- mean tracking error during the active tracking phase,
- mean position error over the full rollout,
- final position error,
- maximum position error,
- number of terminations,
- number of truncations,
- automatic failure-mode classification.

The distinction between mean tracking error and mean position error is important. The mean tracking error is computed only during the active tracking phase of the trajectory. Start-hold and final-hold phases are excluded if they are marked as non-tracking phases. This metric is therefore the main measure for trajectory-following performance. The mean position error is computed over the full evaluation rollout. It also includes stabilization or hold phases where applicable. This metric is useful as an additional indicator of overall rollout behaviour, but it is less specific to active trajectory tracking.

The failure classification can identify issues such as early termination, repeated truncation, action saturation, insufficient motion, overshoot, z-instability, attitude instability or no detected failure.

The evaluation does not rely on a single success-rate number. Instead, it combines tracking-error metrics, termination and truncation counts, and automatic failure-mode classification. Visual evaluation is also important because a low scalar error does not always fully describe the behaviour of the drone. Therefore, trajectory plots and GIFs are used.

## Results

The results are reported in two parts. First, the fixed generalization evaluation is used as the main quantitative comparison between the trained policies. Second, the show scenarios are used as qualitative out-of-distribution stress tests. The tables report aggregated values across all tasks or scenarios of the corresponding evaluation suite.

Lower tracking and position errors indicate better tracking performance. However, the error metrics are not interpreted in isolation. Early termination or truncation can make the mean tracking error appear artificially low because the average is only computed over the completed part of the rollout. Therefore, the completion ratio and the completion-adjusted tracking error are used as additional ranking aids. The completion-adjusted tracking error penalizes policies that track accurately only for a short time but fail before completing the task and is used as the main ranking indicator.

The fixed evaluation summary compares the policies on deterministic tasks that are not identical to the sampled training episodes. This evaluation is used to assess whether a policy learned a generally useful trajectory-tracking behaviour within the considered task families. The show/OOD scenario summary is treated separately because these scenarios are more visual and stress-test oriented. They are useful for identifying instability, overshoot, insufficient motion or action saturation, but they are not used as the primary quantitative benchmark.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown
from IPython.display import display as ipy_display

from src import evaluation

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

runs_root = evaluation.report.find_default_runs_root()


def _display_metric_summary(
    title: str,
    rows: list[dict[str, object]],
    all_columns: list[str],
    compact_columns: list[str],
    empty_message: str,
) -> None:
    ipy_display(Markdown(f"### {title}"))
    report_df = pd.DataFrame(rows, columns=all_columns)
    display_columns = [column for column in compact_columns if column in report_df.columns]
    report_summary = report_df.loc[:, display_columns]
    if report_summary.empty:
        print(empty_message)
    else:
        ipy_display(report_summary)

In [2]:
fixed_rows = evaluation.report.build_aggregated_report_metric_table(runs_root)
_display_metric_summary(
    "Fixed evaluation summary",
    fixed_rows,
    list(evaluation.report.AGGREGATED_REPORT_METRIC_COLUMNS),
    list(evaluation.report.compact_aggregated_report_columns()),
    f"No fixed/generalization metric rows found under {runs_root}.",
)

### Fixed evaluation summary

,run_name,method,action_interface,variant,training_target,evaluated_task_count,mean_tracking_error_m,completion_ratio,completion_adjusted_tracking_error_m,rollout_completion_ratio,mean_position_error_m,final_position_error_m,max_position_error_m,terminated_count,truncated_count,primary_failure
0,llm_curriculum_pid_dynprev_m-taskdist_medium_s...,LLM curriculum,pid_position,default,llm_curriculum,17,0.110642,0.890779,0.123375,1.0,0.118758,0.128348,0.290346,79,7,"attitude_instability, no_failure_detected"
1,curriculum_manual_pid_dynprev_m-taskdist_mediu...,Manual curriculum,pid_position,default,manual_curriculum,17,0.101840,0.831563,0.124564,1.0,0.111932,0.095150,0.290880,77,15,"attitude_instability, no_failure_detected"
2,direct_ppo_directrpm_dynprev_m-taskdist_medium...,Direct PPO,direct_rpm,gamma095,m-taskdist_medium,17,0.124400,0.916231,0.135956,1.0,0.129960,0.133871,0.297864,80,0,"attitude_instability, no_failure_detected"
3,curriculum_manual_directrpm_dynprev_m-taskdist...,Manual curriculum,direct_rpm,default,manual_curriculum,17,0.128716,0.885885,0.146782,1.0,0.133462,0.124919,0.294548,74,10,"attitude_instability, no_failure_detected"
4,llm_curriculum_directrpm_dynprev_m-taskdist_me...,LLM curriculum,direct_rpm,default,llm_curriculum,17,0.135522,0.897654,0.154002,1.0,0.141462,0.138075,0.293461,58,26,"attitude_instability, no_failure_detected"
5,direct_ppo_directrpm_dynprev_m-taskdist_medium...,Direct PPO,direct_rpm,default,m-taskdist_medium,17,0.158233,0.916231,0.173604,1.0,0.159851,0.156673,0.309930,80,0,"attitude_instability, no_failure_detected"
6,direct_ppo_pid_dynprev_m-taskdist_medium_gamma...,Direct PPO,pid_position,gamma095,m-taskdist_medium,17,0.151007,0.862885,0.182992,1.0,0.157928,0.150956,0.319928,75,11,"attitude_instability, no_failure_detected"
7,direct_ppo_directrpm_dynprev_m-taskdist_medium...,Direct PPO,direct_rpm,ent005,m-taskdist_medium,17,0.206405,0.916231,0.226324,1.0,0.206053,0.189955,0.375502,80,0,action_saturation
8,direct_ppo_pid_dynprev_m-taskdist_medium_clip0...,Direct PPO,pid_position,clip010,m-taskdist_medium,17,0.197380,0.870214,0.239139,1.0,0.201786,0.187057,0.435372,65,22,"action_saturation, attitude_instability, no_fa..."
9,direct_ppo_pid_dynprev_m-taskdist_medium_seed0,Direct PPO,pid_position,default,m-taskdist_medium,17,0.205276,0.859809,0.241611,1.0,0.204090,0.209929,0.404392,78,9,"attitude_instability, no_failure_detected"


The fixed evaluation shows that the best-performing policies are mostly curriculum-based. The PID curriculum policies achieve the lowest completion-adjusted tracking errors, while the direct-RPM curriculum and the best direct-RPM baseline remain close behind. This suggests that curriculum learning improves generalization on the fixed evaluation suite, but also that the action interface has a strong influence on the learned behaviour.

The aggregated table alone does not fully explain how the policies behave during tracking. Therefore, selected qualitative rollouts are shown below. For the action-interface comparison, the manual curriculum models are used for both `pid_position` and `direct_rpm`. This keeps the training strategy fixed and highlights the behavioural difference between the two action interfaces. The selected tasks are curved or direction-changing trajectories, because oscillations, phase lag and smoothness differences are easier to see on these tasks than on simple straight lines.

### Qualitative action-interface comparison

To better understand the difference between the two action interfaces, selected fixed-evaluation tasks are visualized qualitatively. The comparison uses the manual curriculum models for both `pid_position` and `direct_rpm`. This keeps the training strategy fixed and highlights the behavioural difference between the two action interfaces.

Curved and direction-changing tasks are selected because oscillations, phase lag and smoothness differences are easier to observe on these trajectories than on simple straight-line tasks.

#### Ellipse

<table style="width:100%; table-layout:fixed;">
<tr>
<td style="width:50%; vertical-align:top; padding:8px;">

<p align="center"><strong>Manual curriculum PID</strong></p>

<img src="docs/media/PID-RPM/curriculum_manual_pid_dynprev_m-taskdist_medium_seed0/ellipse_basic/scenario_rollout.gif" width="100%">

<br>

<img src="docs/media/PID-RPM/curriculum_manual_pid_dynprev_m-taskdist_medium_seed0/ellipse_basic/trajectory_xy.png" width="100%">

</td>
<td style="width:50%; vertical-align:top; padding:8px;">

<p align="center"><strong>Manual curriculum direct RPM</strong></p>

<img src="docs/media/PID-RPM/curriculum_manual_directrpm_dynprev_m-taskdist_medium_seed0/ellipse_basic/scenario_rollout.gif" width="100%">

<br>

<img src="docs/media/PID-RPM/curriculum_manual_directrpm_dynprev_m-taskdist_medium_seed0/ellipse_basic/trajectory_xy.png" width="100%">

</td>
</tr>
</table>

#### Figure-eight

<table style="width:100%; table-layout:fixed;">
<tr>
<td style="width:50%; vertical-align:top; padding:8px;">

<p align="center"><strong>Manual curriculum PID</strong></p>

<img src="docs/media/PID-RPM/curriculum_manual_pid_dynprev_m-taskdist_medium_seed0/figure_eight_basic/scenario_rollout.gif" width="100%">

<br>

<img src="docs/media/PID-RPM/curriculum_manual_pid_dynprev_m-taskdist_medium_seed0/figure_eight_basic/trajectory_xy.png" width="100%">

</td>
<td style="width:50%; vertical-align:top; padding:8px;">

<p align="center"><strong>Manual curriculum direct RPM</strong></p>

<img src="docs/media/PID-RPM/curriculum_manual_directrpm_dynprev_m-taskdist_medium_seed0/figure_eight_basic/scenario_rollout.gif" width="100%">

<br>

<img src="docs/media/PID-RPM/curriculum_manual_directrpm_dynprev_m-taskdist_medium_seed0/figure_eight_basic/trajectory_xy.png" width="100%">

</td>
</tr>
</table>

#### Zigzag

<table style="width:100%; table-layout:fixed;">
<tr>
<td style="width:50%; vertical-align:top; padding:8px;">

<p align="center"><strong>Manual curriculum PID</strong></p>

<img src="docs/media/PID-RPM/curriculum_manual_pid_dynprev_m-taskdist_medium_seed0/zigzag_basic/scenario_rollout.gif" width="100%">

<br>

<img src="docs/media/PID-RPM/curriculum_manual_pid_dynprev_m-taskdist_medium_seed0/zigzag_basic/trajectory_xy.png" width="100%">

</td>
<td style="width:50%; vertical-align:top; padding:8px;">

<p align="center"><strong>Manual curriculum direct RPM</strong></p>

<img src="docs/media/PID-RPM/curriculum_manual_directrpm_dynprev_m-taskdist_medium_seed0/zigzag_basic/scenario_rollout.gif" width="100%">

<br>

<img src="docs/media/PID-RPM/curriculum_manual_directrpm_dynprev_m-taskdist_medium_seed0/zigzag_basic/trajectory_xy.png" width="100%">

</td>
</tr>
</table>

The qualitative comparison shows that the two action interfaces can produce visibly different behaviour even when the same curriculum strategy is used. The PID-based policy is more indirect because the learned action corresponds to target-position commands for a lower-level controller. This can lead to more oscillatory or delayed behaviour on curved trajectories. The direct-RPM policy has more direct authority over the drone motors and can therefore produce a different tracking style, but it is also harder to train because the policy must learn more of the low-level control behaviour itself.

In [3]:
scenario_rows = evaluation.report.build_aggregated_scenario_metric_table(runs_root)
_display_metric_summary(
    "Show / OOD scenario summary",
    scenario_rows,
    list(evaluation.report.AGGREGATED_SCENARIO_REPORT_METRIC_COLUMNS),
    list(evaluation.report.compact_aggregated_scenario_columns()),
    f"No show/OOD scenario metric rows found under {runs_root}.",
)

### Show / OOD scenario summary

,run_name,method,training_target,action_interface,variant,evaluated_scenario_count,mean_tracking_error_m,completion_ratio,completion_adjusted_tracking_error_m,rollout_completion_ratio,mean_position_error_m,final_position_error_m,max_position_error_m,terminated_count,truncated_count,primary_failure
0,curriculum_manual_pid_dynprev_m-taskdist_mediu...,Manual curriculum,manual_curriculum,pid_position,default,3,0.243282,1.000000,0.243282,0.890278,0.244302,0.259410,0.650158,3,0,none
1,llm_curriculum_pid_dynprev_m-taskdist_medium_s...,LLM curriculum,llm_curriculum,pid_position,default,3,0.247098,1.000000,0.247098,0.890278,0.248141,0.274545,0.661364,3,0,none
2,llm_curriculum_directrpm_dynprev_m-taskdist_me...,LLM curriculum,llm_curriculum,direct_rpm,default,3,0.153620,0.545303,0.287225,0.476265,0.159798,0.208312,0.289792,0,3,truncated
3,direct_ppo_pid_dynprev_m-taskdist_medium_gamma...,Direct PPO,m-taskdist_medium,pid_position,gamma095,3,0.305022,1.000000,0.305022,0.890278,0.304908,0.292021,0.708663,3,0,none
4,curriculum_manual_directrpm_dynprev_m-taskdist...,Manual curriculum,manual_curriculum,direct_rpm,default,3,0.135508,0.460602,0.319799,0.398115,0.143799,0.170753,0.324838,0,3,truncated
5,direct_ppo_directrpm_dynprev_basic_show_seed0,Direct PPO,direct_basic_show,direct_rpm,basic_show,3,0.217889,0.724448,0.322287,0.625248,0.221101,0.236008,0.478403,1,2,"none, truncated"
6,direct_ppo_directrpm_dynprev_m-taskdist_medium...,Direct PPO,m-taskdist_medium,direct_rpm,gamma095,3,0.128975,0.423941,0.344483,0.364261,0.138092,0.213077,0.307862,0,3,truncated
7,direct_ppo_directrpm_dynprev_m-taskdist_medium...,Direct PPO,m-taskdist_medium,direct_rpm,default,3,0.192692,0.513031,0.420600,0.436062,0.197317,0.232414,0.399945,0,3,truncated
8,direct_ppo_pid_dynprev_basic_show_seed0,Direct PPO,direct_basic_show,pid_position,basic_show,3,0.376720,0.874224,0.451869,0.766319,0.370989,0.313956,0.713705,2,1,"none, truncated"
9,direct_ppo_pid_dynprev_m-taskdist_medium_seed0,Direct PPO,m-taskdist_medium,pid_position,default,3,0.471291,1.000000,0.471291,0.890278,0.465778,0.443416,0.839594,3,0,none


The show/OOD scenario summary is complemented with selected visual rollouts. The table gives the aggregated performance across the easy, medium and hard scenarios, while the visualizations show how the policies actually behave during the trajectories.

For this qualitative comparison, the two LLM curriculum variants are shown: one with the `pid_position` action interface and one with the `direct_rpm` action interface. This makes it possible to compare how the same curriculum-generation approach behaves with different control interfaces. The easy, medium and hard scenarios are shown separately because the differences between the policies become more visible as the scenario difficulty increases.

The combined GIFs show both rollouts side by side with synchronized playback. In each GIF, the LLM PID policy is shown on the left and the LLM direct-RPM policy is shown on the right. The XY plots show the horizontal tracking behaviour, and the XYZ component plots show how well each spatial coordinate follows the reference over time.

### Qualitative OOD scenario comparison

#### Easy scenario

<p align="center">
  <img src="docs/media/OOD/combined/ood_easy_llm_pid_vs_directrpm.gif" width="100%">
</p>

<table style="width:100%; table-layout:fixed;">
<tr>
<td style="width:50%; vertical-align:top; padding:8px;">

<p align="center"><strong>LLM curriculum PID: XY trajectory</strong></p>

<img src="docs/media/OOD/llm_curriculum_pid_dynprev_m-taskdist_medium_seed0/easy/trajectory_xy.png" width="100%">

</td>
<td style="width:50%; vertical-align:top; padding:8px;">

<p align="center"><strong>LLM curriculum direct RPM: XY trajectory</strong></p>

<img src="docs/media/OOD/llm_curriculum_directrpm_dynprev_m-taskdist_medium_seed0/easy/trajectory_xy.png" width="100%">

</td>
</tr>
<tr>
<td style="width:50%; vertical-align:top; padding:8px;">

<p align="center"><strong>LLM curriculum PID: XYZ components</strong></p>

<img src="docs/media/OOD/llm_curriculum_pid_dynprev_m-taskdist_medium_seed0/easy/trajectory_xyz.png" width="100%">

</td>
<td style="width:50%; vertical-align:top; padding:8px;">

<p align="center"><strong>LLM curriculum direct RPM: XYZ components</strong></p>

<img src="docs/media/OOD/llm_curriculum_directrpm_dynprev_m-taskdist_medium_seed0/easy/trajectory_xyz.png" width="100%">

</td>
</tr>
</table>

#### Medium scenario

<p align="center">
  <img src="docs/media/OOD/combined/ood_medium_llm_pid_vs_directrpm.gif" width="100%">
</p>

<table style="width:100%; table-layout:fixed;">
<tr>
<td style="width:50%; vertical-align:top; padding:8px;">

<p align="center"><strong>LLM curriculum PID: XY trajectory</strong></p>

<img src="docs/media/OOD/llm_curriculum_pid_dynprev_m-taskdist_medium_seed0/medium/trajectory_xy.png" width="100%">

</td>
<td style="width:50%; vertical-align:top; padding:8px;">

<p align="center"><strong>LLM curriculum direct RPM: XY trajectory</strong></p>

<img src="docs/media/OOD/llm_curriculum_directrpm_dynprev_m-taskdist_medium_seed0/medium/trajectory_xy.png" width="100%">

</td>
</tr>
<tr>
<td style="width:50%; vertical-align:top; padding:8px;">

<p align="center"><strong>LLM curriculum PID: XYZ components</strong></p>

<img src="docs/media/OOD/llm_curriculum_pid_dynprev_m-taskdist_medium_seed0/medium/trajectory_xyz.png" width="100%">

</td>
<td style="width:50%; vertical-align:top; padding:8px;">

<p align="center"><strong>LLM curriculum direct RPM: XYZ components</strong></p>

<img src="docs/media/OOD/llm_curriculum_directrpm_dynprev_m-taskdist_medium_seed0/medium/trajectory_xyz.png" width="100%">

</td>
</tr>
</table>

#### Hard scenario

<p align="center">
  <img src="docs/media/OOD/combined/ood_hard_llm_pid_vs_directrpm.gif" width="100%">
</p>

<table style="width:100%; table-layout:fixed;">
<tr>
<td style="width:50%; vertical-align:top; padding:8px;">

<p align="center"><strong>LLM curriculum PID: XY trajectory</strong></p>

<img src="docs/media/OOD/llm_curriculum_pid_dynprev_m-taskdist_medium_seed0/hard/trajectory_xy.png" width="100%">

</td>
<td style="width:50%; vertical-align:top; padding:8px;">

<p align="center"><strong>LLM curriculum direct RPM: XY trajectory</strong></p>

<img src="docs/media/OOD/llm_curriculum_directrpm_dynprev_m-taskdist_medium_seed0/hard/trajectory_xy.png" width="100%">

</td>
</tr>
<tr>
<td style="width:50%; vertical-align:top; padding:8px;">

<p align="center"><strong>LLM curriculum PID: XYZ components</strong></p>

<img src="docs/media/OOD/llm_curriculum_pid_dynprev_m-taskdist_medium_seed0/hard/trajectory_xyz.png" width="100%">

</td>
<td style="width:50%; vertical-align:top; padding:8px;">

<p align="center"><strong>LLM curriculum direct RPM: XYZ components</strong></p>

<img src="docs/media/OOD/llm_curriculum_directrpm_dynprev_m-taskdist_medium_seed0/hard/trajectory_xyz.png" width="100%">

</td>
</tr>
</table>

The visual comparison shows that the two LLM curriculum policies behave differently despite being generated by the same curriculum mechanism. The PID-based policy is more stable and completes the scenarios more reliably. However, the XYZ component plots are important for checking whether all three coordinates are actively tracked, rather than only whether the drone remains stable.

The direct-RPM policy is harder to train and completes less of the OOD scenarios, but it can show more direct tracking behaviour because the policy acts closer to the motor-control level. This illustrates why the OOD table and the visual rollouts must be interpreted together: the completion-aware metrics indicate how much of the scenario was completed, while the GIFs and trajectory plots show whether the drone actually follows the reference throughout the trajectory.

### Task-specific basic-show behaviour

The `basic_show` runs are treated separately from the main fixed-generalization and OOD comparisons. Unlike the `tracking_medium` runs, the `basic_show` policy is trained on one single connected show-style trajectory. The following comparison shows the PID-based `basic_show` policy on its own trained show trajectory on the left and on the medium show scenario on the right.

<p align="center">
  <img src="docs/media/direct_ppo_pid_dynprev_basic_show_seed0/basic_show_vs_medium_combined_short.gif" width="100%">
</p>

The comparison illustrates a task-specific specialization effect. On the trained show-style trajectory, the policy reproduces behaviour that is closely adapted to the repeated movement pattern. When the same policy is evaluated on the medium show scenario, this behaviour does not transfer in the same way. In the PID case, the controller appears to adapt strongly to the trained motion sequence, which can be interpreted as task-specific overfitting rather than robust general trajectory tracking.

For this reason, the `basic_show` runs are useful as qualitative examples, but they are not used as the main evidence for generalization performance.

## Discussion
The results show that curriculum learning is beneficial for this drone trajectory-tracking task. The best fixed-generalization results are achieved by curriculum-based policies, while the strongest direct PPO baselines remain competitive but are generally less robust. This suggests that the task is difficult to learn directly from the full randomized task distribution and that a staged progression helps the policy learn more stable tracking behaviour.

The comparison between manual and LLM-guided curricula shows an important difference. A manual curriculum can work well, but it requires careful hand-design. The progression that works for one action interface does not necessarily transfer directly to another one. The PID interface provides a higher-level control abstraction and is easier to stabilize, while the direct-RPM interface exposes the policy more directly to the motor-level dynamics. Therefore, the curriculum has to match not only the task difficulty, but also the control interface. The LLM-guided curriculum is useful in this context because it can propose a progression that adapts to the observed training behaviour.

The action-interface comparison also shows that good scalar metrics are not sufficient to fully understand the learned behaviour. The PID-based policies are generally more stable, but their trajectories can be more oscillatory or delayed. In some cases, the drone remains stable but does not actively follow all trajectory components equally well. The direct-RPM policies can produce smoother and more direct tracking behaviour, but they are harder to train and more prone to overshoot, truncation or instability. This reflects the trade-off between a higher-level action interface that hides part of the low-level control problem and a lower-level interface that gives the policy more direct control authority.

The PPO variant comparison shows that hyperparameters have a strong influence on the final behaviour. The discount factor was especially important. A smaller value, such as `gamma095`, improved the performance of some direct PPO baselines. One possible explanation is that the shorter effective planning horizon makes the policy focus more strongly on immediate tracking and stabilization instead of optimizing long-horizon returns that may be harder to estimate reliably in this dynamic control task. However, this effect is not universal across all settings, so the discount factor should be treated as a sensitive design parameter rather than a generally optimal fixed value.

Model capacity was also important. Early experiments with smaller networks showed that the policy network must be large enough to represent the drone dynamics and the different trajectory-following behaviours. A network that is too small can fail to learn the task at all. At the same time, simply increasing the network size is not sufficient on its own. The task formulation, action interface, curriculum design and PPO hyperparameters all interact with each other.

Another important observation is that the way tasks are formulated strongly affects what the policy learns. Training on a single connected show-style trajectory can lead to task-specific specialization. This supports the observation that task variation is necessary if the goal is not only to reproduce one trajectory, but to learn a more general tracking behaviour.

There are also limitations. The experiments are performed in simulation and use a limited number of seeds. Therefore, the results should be interpreted as evidence from this project setup rather than as a final general statement about all drone-control tasks. The LLM is used only as a curriculum proposer and not as a controller, which makes the approach safer and easier to validate, but also limits what the LLM can influence. More training runs, more random seeds and a broader hyperparameter search would be needed for a stronger statistical conclusion.

## Conclusion and Outlook
The results provide evidence that curriculum learning can improve PPO-based drone trajectory tracking compared to direct training on the full task distribution within the investigated simulation setup. They also demonstrate that a local LLM can be integrated as a safe curriculum task proposer when its outputs are restricted to structured task definitions and validated deterministically before training.

The results also show that curriculum learning alone is not sufficient. The final behaviour depends strongly on the task formulation, PPO hyperparameters, model capacity and action interface. In particular, the difference between `pid_position` and `direct_rpm` was important. The PID interface was easier to stabilize, while the direct-RPM interface allowed more direct motor-level control but was more difficult to train and more prone to overshoot or instability.

A frequent issue during evaluation was action saturation, where the policy repeatedly selected actions close to the limits of the action space. This can indicate that the policy is trying to correct too aggressively or that the task, reward or controller configuration leads to overly strong control commands. Future work could therefore investigate stronger action-smoothness penalties, better action regularization or modified reward terms that discourage persistent saturation without preventing the drone from reacting quickly when necessary.

For the PID-based interface, another possible improvement would be to tune the lower-level PID controller more carefully. A less aggressive PID configuration could reduce overshoot and oscillatory behaviour. This would make the high-level PPO task more stable and could improve the interpretation of the learned policy, because fewer effects would come from the underlying controller dynamics.

For the LLM-guided curriculum, future work could separate diagnostic interpretation from task proposal. The current implementation already provides structured curriculum feedback, metrics, history and diagnostic signals to the LLM. A possible extension would be to use a first LLM step to interpret these diagnostics and produce a structured recovery plan, for example identifying action saturation, insufficient motion, overshoot or z-instability as the dominant issue. A second LLM step could then convert this recovery plan into a concrete curriculum task proposal within the validated task schema. This would keep the system constrained and safe, while making the curriculum adaptation more systematic and easier to inspect.

The current project is limited to a single drone in simulation. A natural next step would be to extend the setup to multi-drone trajectory tracking or formation control. This would make the task more complex because the agents would have to track references while also avoiding collisions or maintaining relative positions. It would also be interesting to investigate whether curriculum learning and LLM-guided task proposals become more useful as the task complexity increases.

Finally, the experiments should be repeated with more random seeds and a broader hyperparameter search. This would make the comparison statistically stronger and would help separate robust effects from seed-specific behaviour. Additional future work could compare PPO with other continuous-control algorithms, add domain randomization and investigate whether the learned policies can transfer from simulation to real drone hardware.

## References

- Schulman, J., Wolski, F., Dhariwal, P., Radford, A., and Klimov, O. “Proximal Policy Optimization Algorithms.” arXiv:1707.06347, 2017.
- Raffin, A., Hill, A., Gleave, A., Kanervisto, A., Ernestus, M., and Dormann, N. “Stable-Baselines3: Reliable Reinforcement Learning Implementations.” Journal of Machine Learning Research, 2021.
- Panerati, J., Zheng, H., Zhou, S., Xu, J., Prorok, A., and Schoellig, A. P. “Learning to Fly: A Gym Environment with PyBullet Physics for Reinforcement Learning of Multi-agent Quadcopter Control.” 2021.
- Coumans, E. and Bai, Y. PyBullet, a Python module for physics simulation.
- Farama Foundation. Gymnasium documentation and API.

ChatGPT by OpenAI was used for linguistic editing, report structuring and support with code snippets.  
The author remains fully responsible for the technical correctness, implementation and final content of this work.

## Declaration of Independence
I hereby certify that I have written this thesis independently and have not used any auxiliary materials other than those indicated. The passages of the work, which are taken in the wording or the sense after other works (to it also Internet sources count), were marked under indication of the source.

<table style="width:100%; background-color: white; padding: 10px; border-radius: 6px; box-shadow: 0 0 5px rgba(0,0,0,0.2); margin-top:20px;">
  <tr>
    <td align="left">
      <img src="docs/images/Unterschrift.png" alt="Unterschrift" style="height:150px;">
    </td>
  </tr>
</table>

---
---

# Appendix

## 🎞️ Evaluation Rollout Gallery

The following rollout gallery shows the fixed generalization tasks for the model `direct_ppo_directrpm_dynprev_m-taskdist_medium_gamma095_seed0`. This model is used as a representative direct-PPO baseline because it achieved the strongest fixed-evaluation performance among the direct PPO variants. The gallery is included as an appendix to provide visual examples of the evaluation task suite.

<details>
<summary><strong>Stabilization and altitude tasks</strong></summary>

<br>

<table>
  <tr>
    <td align="center" width="33%">
      <strong>Hover center</strong><br>
      <img src="docs/media/generalization_direct_ppo_directrpm_dynprev_m-taskdist_medium_gamma095_seed0/hover_center/scenario_rollout.gif" width="400">
    </td>
    <td align="center" width="33%">
      <strong>Vertical basic</strong><br>
      <img src="docs/media/generalization_direct_ppo_directrpm_dynprev_m-taskdist_medium_gamma095_seed0/vertical_basic/scenario_rollout.gif" width="400">
    </td>
    <td align="center" width="33%">
      <strong>Angled descent</strong><br>
      <img src="docs/media/generalization_direct_ppo_directrpm_dynprev_m-taskdist_medium_gamma095_seed0/angled_descent_basic/scenario_rollout.gif" width="400">
    </td>
  </tr>
</table>

</details>

<details>
<summary><strong>Line and altitude-tracking tasks</strong></summary>

<br>

<table>
  <tr>
    <td align="center" width="33%">
      <strong>Line basic</strong><br>
      <img src="docs/media/generalization_direct_ppo_directrpm_dynprev_m-taskdist_medium_gamma095_seed0/line_basic/scenario_rollout.gif" width="400">
    </td>
    <td align="center" width="33%">
      <strong>Diagonal line</strong><br>
      <img src="docs/media/generalization_direct_ppo_directrpm_dynprev_m-taskdist_medium_gamma095_seed0/diagonal_line_basic/scenario_rollout.gif" width="400">
    </td>
    <td align="center" width="33%">
      <strong>Short line start hold</strong><br>
      <img src="docs/media/generalization_direct_ppo_directrpm_dynprev_m-taskdist_medium_gamma095_seed0/short_line_start_hold/scenario_rollout.gif" width="400">
    </td>
  </tr>
  <tr>
    <td align="center" width="33%">
      <strong>Delayed altitude polyline</strong><br>
      <img src="docs/media/generalization_direct_ppo_directrpm_dynprev_m-taskdist_medium_gamma095_seed0/delayed_altitude_polyline_basic/scenario_rollout.gif" width="400">
    </td>
    <td align="center" width="33%">
      <strong>Multi-height polyline</strong><br>
      <img src="docs/media/generalization_direct_ppo_directrpm_dynprev_m-taskdist_medium_gamma095_seed0/multi_height_polyline_basic/scenario_rollout.gif" width="400">
    </td>
    <td align="center" width="33%">
    </td>
  </tr>
</table>

</details>

<details>
<summary><strong>Polyline and corner tasks</strong></summary>

<br>

<table>
  <tr>
    <td align="center" width="33%">
      <strong>L-shape polyline</strong><br>
      <img src="docs/media/generalization_direct_ppo_directrpm_dynprev_m-taskdist_medium_gamma095_seed0/polyline_l_basic/scenario_rollout.gif" width="400">
    </td>
    <td align="center" width="33%">
      <strong>Rectangle</strong><br>
      <img src="docs/media/generalization_direct_ppo_directrpm_dynprev_m-taskdist_medium_gamma095_seed0/rectangle_basic/scenario_rollout.gif" width="400">
    </td>
    <td align="center" width="33%">
      <strong>Square</strong><br>
      <img src="docs/media/generalization_direct_ppo_directrpm_dynprev_m-taskdist_medium_gamma095_seed0/square_basic/scenario_rollout.gif" width="400">
    </td>
  </tr>
  <tr>
    <td align="center" width="33%">
      <strong>Triangle</strong><br>
      <img src="docs/media/generalization_direct_ppo_directrpm_dynprev_m-taskdist_medium_gamma095_seed0/triangle_basic/scenario_rollout.gif" width="400">
    </td>
    <td align="center" width="33%">
      <strong>Zigzag</strong><br>
      <img src="docs/media/generalization_direct_ppo_directrpm_dynprev_m-taskdist_medium_gamma095_seed0/zigzag_basic/scenario_rollout.gif" width="400">
    </td>
    <td align="center" width="33%">
    </td>
  </tr>
</table>

</details>

<details>
<summary><strong>Curved trajectory tasks</strong></summary>

<br>

<table>
  <tr>
    <td align="center" width="33%">
      <strong>Circle</strong><br>
      <img src="docs/media/generalization_direct_ppo_directrpm_dynprev_m-taskdist_medium_gamma095_seed0/circle_basic/scenario_rollout.gif" width="400">
    </td>
    <td align="center" width="33%">
      <strong>Ellipse</strong><br>
      <img src="docs/media/generalization_direct_ppo_directrpm_dynprev_m-taskdist_medium_gamma095_seed0/ellipse_basic/scenario_rollout.gif" width="400">
    </td>
    <td align="center" width="33%">
      <strong>Figure eight</strong><br>
      <img src="docs/media/generalization_direct_ppo_directrpm_dynprev_m-taskdist_medium_gamma095_seed0/figure_eight_basic/scenario_rollout.gif" width="400">
    </td>
  </tr>
</table>

</details>